# Cross-analyse ten EIGER flat-field corrections

Extract independent pixel/sensor divisor maps from **100207–100216**, then apply each map to all ten scans: **100 corrected scan/correction combinations**, plus ten physical-only controls. Raw scans and their companion files are read only from `/dls/i15-1/data/2026/cm44163-4`. Geometry is taken from each scan's embedded JSON, not a different calibration file.

Select the kernel using `/dls/science/users/fer45166/envs/pyfai_new/bin/python`, then run top-to-bottom. The settings cell verifies the interpreter. All run products, caches, geometry temporary files, logs and plots are placed under this repository's `outputs/cross_analysis/<RUN_NAME>/`; the facility directory is never written. Existing completed fits/integrations are reused only if raw-file metadata, settings and implementation fingerprints match. Change `RUN_NAME` when changing the experiment. Interrupted `.partial.h5` files can be regenerated safely.

## What makes a correction “best”?

The primary metric measures **internal consistency between neighbouring contributing frames**, not agreement with its own training pattern. For each supported overlapping bin, calculate $r=\log(I_{\mathrm{left}}/I_{\mathrm{right}})$. The frame-pair shape error is $100\sqrt{\operatorname{mean}[(r-\operatorname{median}(r))^2]}$; take the median across frame pairs. These are approximate fractional percent errors, not exact linear percentages. The median offset is removed **only for scoring**, never used to renormalize or modify exported intensity. Also report uncentered overlap, offsets and counting-noise-scaled residuals.

Rank each correction by its **median percent reduction in centered overlap versus the same target's physical-only control, on the nine other scans**. The ten diagonal self-applications are useful training diagnostics, but excluded from rankings. Positive improvement is better; a negative value means the correction made consistency worse. The 100 combinations are not 100 independent replicates.

### Fairness safeguards
- Every combination and baseline uses the **same fixed union of native masks plus detector boundaries**. Learned bad-pixel masks are deliberately **not** used in this primary benchmark: they could make a donor look better just by discarding difficult pixels, and a union of learned masks would let each target's own fit select its validation pixels. Saved calibration files still contain their learned masks for normal processing.
- Each target uses its own measured I0. **No source-frame gains are transferred, even for self-applications.** No target-pattern residual rejection, smoothing, or detector-map refitting is performed while applying corrections.
- Geometry, wavelength, energy, material, thickness and orientation must match. Comparison is restricted to the fitted q interval; full-range 2theta extrapolation is intentionally not part of this ranking.
- The raw detector is read one frame at a time; fitting holds only one scan cache at a time. Each target's 11 integrations are saved in one compact HDF5 checkpoint.

### Limits of the conclusions
Overlap consistency is not an absolute flat-field accuracy measurement: sample evolution, geometry error, I0 error and physical-model error can confound it. A correction can improve centered shape while harming absolute normalization, so inspect both metrics. Repeat-to-repeat shape agreement and correction-map/sensor-gain stability are secondary diagnostics, not ground truth. Fitting and integrating use the same physical assumptions as the existing notebooks (including modeled CdTe absorption, not measured QE).

Bootstrap intervals resample **whole target scans**, retaining their paired corrections, not pixels/bins. There are only ten related repeats and calibration donors overlap the evaluation set; intervals and best-rank frequencies are exploratory, not independent certification or formal significance tests. Pairwise comparisons of the selected winner also have selection bias. Counting sigma excludes calibration/I0/model uncertainty and neighbouring-bin covariance.

**Outputs:** ten reusable calibration HDF5 files; ten target HDF5 files holding all 100 corrected profiles plus ten baselines and per-frame intensity/sigma/count arrays; `pair_metrics_100.csv`, `baseline_metrics_10.csv`, `correction_ranking.csv`, `paired_best_comparisons.csv`, `summary.json`, provenance/environment JSON, fit logs and PNG figures.

In [ ]:
# Settings, local-only temporary files and interpreter check.
from pathlib import Path
import os
import sys
import tempfile

EXPECTED_PYTHON = Path('/dls/science/users/fer45166/envs/pyfai_new/bin/python')
if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(f'Select {EXPECTED_PYTHON} as the notebook kernel; current executable: {sys.executable}')

REPO_DIRECTORY = Path.cwd().resolve()
while REPO_DIRECTORY != REPO_DIRECTORY.parent and not (REPO_DIRECTORY / 'eiger_cross_analysis.py').exists():
    REPO_DIRECTORY = REPO_DIRECTORY.parent
if not (REPO_DIRECTORY / 'eiger_cross_analysis.py').exists():
    raise FileNotFoundError('Launch the notebook from the repository or one of its subdirectories.')
DATA_DIRECTORY = Path('/dls/i15-1/data/2026/cm44163-4').resolve()
SCAN_NUMBERS = list(range(100207, 100217))
RUN_NAME = 'ten_repeats_100207_100216_v1'  # use a new name after changing scientific settings
RUN_DIRECTORY = (REPO_DIRECTORY / 'outputs' / 'cross_analysis' / RUN_NAME).resolve()
if not RUN_DIRECTORY.is_relative_to(REPO_DIRECTORY) or RUN_DIRECTORY.is_relative_to(DATA_DIRECTORY):
    raise ValueError('All outputs must stay in this repository and outside the raw-data directory.')
TEMP_DIRECTORY = RUN_DIRECTORY / 'tmp'
TEMP_DIRECTORY.mkdir(parents=True, exist_ok=True)
for variable in ('TMPDIR', 'TEMP', 'TMP'):
    os.environ[variable] = str(TEMP_DIRECTORY)
tempfile.tempdir = str(TEMP_DIRECTORY)
for variable, subdirectory in (('MPLCONFIGDIR', 'matplotlib'), ('XDG_CACHE_HOME', 'cache'), ('PYTHONPYCACHEPREFIX', 'pycache')):
    directory = TEMP_DIRECTORY / subdirectory
    directory.mkdir(exist_ok=True)
    os.environ[variable] = str(directory)
sys.pycache_prefix = str(TEMP_DIRECTORY / 'pycache')
# Keep CPU execution predictable and avoid thread oversubscription on the workstation.
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['OPENBLAS_NUM_THREADS'] = '4'

SETTINGS = {
    'polarization': 0.95, 'density': 6.2, 'saturation': 1e7,
    'min_count': 100, 'min_bins': 20,
    'fit': {
        'q_range': [200.0, 400.0], 'npt': 800, 'iterations': 10,
        'fit_frame_gains': True, 'fractional_floor': 0.003,
        'reject_z': 5.0, 'minimum_observations': 8,
        'shrink_weight': 300.0, 'frame_prior_sigma': 0.01,
    },
}
BOOTSTRAP_SAMPLES = 5000
BOOTSTRAP_SEED = 20261012
print('Python:', sys.executable)
print('Version:', sys.version)
print('Read-only raw inputs:', DATA_DIRECTORY)
print('All generated files:', RUN_DIRECTORY)
print('Scan numbers:', SCAN_NUMBERS)

In [ ]:
# Imports, environment metadata and complete ten-scan preflight.
import json
import importlib
import importlib.metadata
import numpy as np
import h5py
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

if str(REPO_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(REPO_DIRECTORY))
import eiger_workflow as workflow
import eiger_cross_analysis as cross
importlib.reload(workflow)
importlib.reload(cross)

ENVIRONMENT = {
    'python_executable': sys.executable, 'python_version': sys.version,
    'packages': {package: importlib.metadata.version(package)
                 for package in ('numpy', 'scipy', 'h5py', 'matplotlib', 'pyFAI', 'xraylib')},
    'temporary_directory': str(TEMP_DIRECTORY),
}
np.testing.assert_allclose(np.mean([1.0, 2.0, 3.0]), 2.0)
scans, geometry_configs, run_fingerprint = cross.prepare(
    DATA_DIRECTORY, SCAN_NUMBERS, RUN_DIRECTORY, REPO_DIRECTORY, SETTINGS,
)
(RUN_DIRECTORY / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2))
print(json.dumps(ENVIRONMENT, indent=2))
print('\nScan     Frames   Angle range (deg)      I0 median       Native mask pixels')
for number, scan in scans.items():
    print(f'{number}   {len(scan.angles):5d}   {scan.angles.min():8.3f}–{scan.angles.max():8.3f} '
          f'  {np.median(scan.i0):13.6g}   {scan.native_mask.sum():8d}')
print('Geometry/detector/energy compatibility passed; run fingerprint:', run_fingerprint)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for number, scan in scans.items():
    axes[0].plot(scan.angles, scan.i0 / np.median(scan.i0), label=str(number), alpha=.7)
    axes[1].plot(scan.angles, scan.i0, alpha=.7)
axes[0].set(xlabel='Scan angle (degrees)', ylabel='I0 / scan median', title='Relative monitor stability')
axes[1].set(xlabel='Scan angle (degrees)', ylabel='Measured I0', title='Monitor scale between repeats')
axes[0].legend(fontsize=7, ncol=2)
fig.savefig(RUN_DIRECTORY / 'figures' / 'input_monitors.png', dpi=160)
plt.show()

## 1. Extract all ten detector maps

This is the first expensive step. Each scan is fitted separately on all of its contributing positions using the original robust pixel/sensor/frame model. Frame gains help separate source-frame effects during fitting but remain **diagnostics only**, not transferable corrections. Independent validation is supplied by the nine other scans in the next stage, rather than repeating the within-scan fold fits here.

Calibration outputs use the existing reusable HDF5 schema, including geometry, physical parameters, support/outlier maps, source-frame gains and convergence history. Detailed iteration logs go to this run's local `logs/` directory. Existing completed calibrations are reused on rerun; you can interrupt between scans without losing earlier fits. Insufficient sensor/pixel support fails explicitly rather than silently lowering thresholds.

In [ ]:
calibration_paths = cross.extract_all(scans, geometry_configs, SETTINGS, RUN_DIRECTORY, run_fingerprint)
assert len(calibration_paths) == 10
for number, path in calibration_paths.items():
    print(number, '->', path.relative_to(REPO_DIRECTORY))

# Summarize support, rejection and convergence before interpreting any ranking.
fit_diagnostics = []
for number, path in calibration_paths.items():
    with h5py.File(path, 'r') as file:
        good = ~file['corrections/base_mask'][()].astype(bool)
        observations = file['corrections/observations'][()]
        history = json.loads(file['fit_diagnostics'].attrs['history_json'])
        gains = file['fit_diagnostics/frame_gains'][()]
        indices = file['fit_diagnostics/fit_indices'][()]
        row = {
            'source_scan': number, 'fit_positions': len(indices),
            'pixels_with_minimum_observations_percent': float(100 * np.mean(observations[good] >= SETTINGS['fit']['minimum_observations'])),
            'learned_bad_pixels': int(file['corrections/bad_pixel_mask'][()].sum()),
            'final_rejected_percent': history[-1]['rejected_percent'],
            'final_max_log_gain_change': history[-1]['max_log_gain_change'],
            'source_frame_gain_log_rms_percent': float(100 * np.sqrt(np.mean(np.log(gains[indices])**2))),
        }
        fit_diagnostics.append(row)
        print(row)
cross.write_csv(RUN_DIRECTORY / 'fit_diagnostics_10.csv', fit_diagnostics)
print('Review final gain changes: ten iterations is a configured limit, not proof of convergence.')

## 2. Apply the ten maps to every scan (100 combinations + ten controls)

All corrected patterns use the fitted q interval and 800 bins by default. Counts are divided once by the selected map; raw-count Poisson variance is divided by its square. Frames are merged using summed signal/normalization/variance accumulators, not an unweighted average of frame intensities. The common native/boundary mask and raw invalid/saturation rejection are unchanged across maps.

This is the other expensive step: 11 integrations per raw frame. One completed checkpoint is written per target scan. In each target HDF5, correction index **0** is the physical-only control (`correction_scan=-1`); indices **1–10** follow `SCAN_NUMBERS`. `frame_intensity`, `frame_sigma` and `frame_count` have axes `(correction, frame, q)`. Merged profiles have axes `(correction, q)`. No full detector images or full-range 2theta products are duplicated 100 times.

In [ ]:
common_mask = cross.compare_all(
    scans, geometry_configs, calibration_paths, SETTINGS, RUN_DIRECTORY, run_fingerprint,
)
print('Fixed benchmark mask:', common_mask.sum(), 'excluded pixels;', (~common_mask).sum(), 'usable pixels')
print('All 100 map/scan combinations and ten physical-only controls are saved.')
print('Source-frame gains were never applied; learned bad-pixel masks were not used to select evaluation pixels.')

## 3. Rank corrections using only non-source targets

The table includes paired improvement versus each target's control; absolute and worst-target overlap; targets improved; per-target wins; self-score; and exploratory target-cluster bootstrap intervals/best-rank fractions. Each correction has nine non-source targets. For direct comparisons with the selected winner, use only the **eight shared targets**, excluding both donors.

`noise_scaled_rms` is descriptive: because split q bins are correlated and the fitted correction is uncertain, it is **not a reduced chi-square or a calibrated p-value**. Repeat-shape agreement in the pair table uses a same-correction median reference excluding both the target and the correction donor; it is also descriptive and can detect sample change as well as calibration problems.

In [ ]:
analysis = cross.analyse(scans, RUN_DIRECTORY, run_fingerprint, BOOTSTRAP_SAMPLES, BOOTSTRAP_SEED)
assert len(analysis['rows']) == 100
assert sum(not row['self_application'] for row in analysis['rows']) == 90
assert all(row['independent_targets'] == 9 for row in analysis['ranking'])
assert all(row['shared_independent_targets'] == 8 for row in analysis['pairwise'])

headers = ['Source', 'Improvement %', 'Exploratory 95% interval', 'Overlap %', 'Worst %', 'Better / 9', 'Wins', 'Bootstrap best fraction']
lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
for row in analysis['ranking']:
    lines.append(f"| {row['correction_scan']} | {row['median_improvement_percent']:.3f} | "
                 f"[{row['bootstrap_low_percent']:.3f}, {row['bootstrap_high_percent']:.3f}] | "
                 f"{row['median_overlap_percent']:.4f} | {row['worst_overlap_percent']:.4f} | "
                 f"{row['targets_better_than_baseline']} | {row['target_wins']} | {row['bootstrap_best_fraction']:.3f} |")
display(Markdown('\n'.join(lines)))
best = analysis['ranking'][0]
if best['median_improvement_percent'] <= 0:
    display(Markdown('**No correction improves the median independent-target shape consistency over physical-only processing.** Do not recommend the top-ranked map merely because it ranks first.'))
else:
    display(Markdown(f"**Best ranked donor: {best['correction_scan']}**, median independent-target improvement "
                     f"**{best['median_improvement_percent']:.2f}%**, improving "
                     f"**{best['targets_better_than_baseline']}/9** targets. This is an exploratory ranking, not an absolute-accuracy certificate."))
print('\nPaired comparisons against the selected winner (other minus best overlap percentage points; positive favours best):')
for row in analysis['pairwise']:
    print(row)
print('\nAll 100 pairs, baselines, rankings and paired comparisons were exported as CSV; summary.json records the bootstrap setup.')

## 4. Visual diagnostics

Heatmaps show all 100 combinations; hatched diagonal entries are not validation. Inspect the ranking intervals together with target-by-target scores: a small median advantage may not transfer consistently. The map-distance and sensor-gain plots test calibration reproducibility. The final patterns show the top-ranked map applied to its nine non-source scans and the change from physical-only processing; those changes are not residuals against ground truth.

Use `pair_metrics_100.csv` for absolute-overlap, offset, counting-noise, overlap-support and repeat-shape diagnostics. If a winner is within the exploratory uncertainty of alternatives, report a group of similarly performing corrections rather than a unique best map.

In [ ]:
figures = cross.plot_analysis(analysis, calibration_paths, RUN_DIRECTORY)
plt.show()
print('Saved figures:', [name + '.png' for name, _ in figures])

# Additional normalization/repeatability diagnostics make primary-score tradeoffs visible.
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for j, number in enumerate(SCAN_NUMBERS):
    rows = [row for row in analysis['rows'] if row['correction_scan'] == number and not row['self_application']]
    axes[0].scatter([row['centered_overlap_percent'] for row in rows],
                    [row['absolute_overlap_percent'] for row in rows], label=str(number), alpha=.7)
    axes[1].scatter([row['target_scan'] for row in rows],
                    [row['leave_target_and_donor_out_repeat_shape_percent'] for row in rows], label=str(number), alpha=.7)
axes[0].set(xlabel='Centered overlap (%)', ylabel='Absolute overlap (%)', title='Shape consistency versus normalization consistency')
axes[0].legend(fontsize=7, ncol=2)
axes[1].set(xlabel='Target scan', ylabel='Repeat shape log-RMS (%)', title='Target/donor-excluded repeat agreement (secondary)')
axes[1].ticklabel_format(axis='x', style='plain', useOffset=False)
fig.savefig(RUN_DIRECTORY / 'figures' / 'absolute_and_repeat_diagnostics.png', dpi=160)
plt.show()

best_path = calibration_paths[analysis['best_number']]
print('Top-ranked reusable calibration:', best_path)
print('Benchmark results:', RUN_DIRECTORY)
print('For normal processing, use this calibration with process_eiger_scan.ipynb after reviewing the independent-target diagnostics.')
print('Its normal learned-mask processing is not identical to this fixed-mask benchmark; retain that distinction in reporting.')